# Capstone — Modern Data Engineering for AI Systems
**SDAIA Academy, delivered via Learning Space | Trainer: Mohammed Albeladi**

This notebook is the executable, single-run entry point for the capstone pipeline that
integrates all five days of the course:

| Stage | Rubric deliverable | Points |
|---|---|---|
| Ingestion | Kafka + Pydantic data contract + quarantine | 20 |
| Delta Lakehouse | Bronze/Silver/Gold, real `MERGE`, schema enforcement | 25 |
| RAG Pipeline | Chunking, ChromaDB, BM25, RRF fusion, cross-encoder rerank | 25 |
| Orchestration | Airflow DAG wiring every stage | 15 |
| Quality Gate + Lineage | Great Expectations gate + OpenLineage events | 15 |

All application code lives in `src/` in this repository so it's reused identically by this
notebook, by `pytest`, and by the Airflow DAG in `src/orchestration/capstone_dag.py` — this
notebook is a thin runner, not a second copy of the logic.

**Run top to bottom in Google Colab.** Two cells (Kafka broker bootstrap, and the RAG
embedding models) need real internet access to download a JVM broker and two ~40MB model
files on first run — Colab has both Java and unrestricted internet, which is why the course's
own Day 2 lab uses exactly this pattern.


In [ ]:
# If running in Colab, clone the repo (skip if you already have it open locally).
import os
if not os.path.exists("src"):
    !git clone https://github.com/Emad404/capstone-modern-data-engineering.git repo
    %cd repo

!pip install -q -r requirements.txt


## Stage 1 — Ingestion: Kafka producer/consumer + Pydantic data contract
Boots a real, local, single-broker Kafka (KRaft mode) exactly as in the Day 2 lab's 'Real Kafka Round Trip' section, then runs the capstone's own producer/consumer against it.

In [ ]:
# Real local Kafka broker (Colab has Java preinstalled). ~15-20s to finish starting.
!pip install -q kafka-python
!curl -sSOL https://downloads.apache.org/kafka/3.7.0/kafka_2.13-3.7.0.tgz && tar -xzf kafka_2.13-3.7.0.tgz
!cd kafka_2.13-3.7.0 && bin/kafka-storage.sh format -t $(bin/kafka-storage.sh random-uuid) -c config/kraft/server.properties
!cd kafka_2.13-3.7.0 && nohup bin/kafka-server-start.sh config/kraft/server.properties > /tmp/kafka.log 2>&1 &
import time; time.sleep(20)
print("Kafka broker should be up — check /tmp/kafka.log if the next cell can't connect.")


In [ ]:
from src.ingestion.kafka_producer import produce_sample_events
produce_sample_events()


In [ ]:
from src.ingestion.kafka_consumer import consume_and_gate
valid_events, quarantined = consume_and_gate()


## Stage 2 — Delta Lakehouse: Bronze / Silver / Gold with a real `MERGE`
Needs the Delta Lake JVM jar from Maven Central — unrestricted in Colab, unlike a locked-down sandbox.

In [ ]:
!pip install -q pyspark==3.5.0 delta-spark==3.2.0
from src.lakehouse.delta_pipeline import run_lakehouse_demo
from src.ingestion.contracts import validate_event
from src.ingestion.kafka_producer import SAMPLE_EVENTS

valid = [validate_event(e, "orders_raw")[0].model_dump() for e in SAMPLE_EVENTS if validate_event(e, "orders_raw")[0]]
wave1 = valid[:5]
wave2 = []
for e in valid[:2]:
    c = dict(e); c["unit_price"] = round(c["unit_price"] * 0.9, 2); wave2.append(c)
wave2 += valid[5:]

run_lakehouse_demo(wave1, wave2)


## Stage 3 — RAG: chunking → hybrid search (dense + BM25 via RRF) → cross-encoder rerank → cited answer
First call downloads `all-MiniLM-L6-v2` and `cross-encoder/ms-marco-MiniLM-L-6-v2` from Hugging Face (~90MB total).

In [ ]:
!pip install -q chromadb sentence-transformers rank-bm25
from src.rag.rag_pipeline import (
    DOCUMENTS, chunk_documents, BM25Index, build_vector_index, dense_search,
    reciprocal_rank_fusion, rerank, answer_with_citations,
)

chunks = chunk_documents(DOCUMENTS, chunk_size=2)
print(f"{len(DOCUMENTS)} documents -> {len(chunks)} chunks")

bm25 = BM25Index(chunks)
collection = build_vector_index(chunks)

query = "How does the pipeline halt when data quality checks fail?"
bm25_hits = bm25.search(query, top_k=6)
vector_hits = dense_search(collection, query, top_k=6)
fused = reciprocal_rank_fusion(vector_hits, bm25_hits, top_k=6)
reranked = rerank(query, fused, top_k=3)

print("\nTop reranked chunks:")
for r in reranked:
    print(f"  score={r['rerank_score']:.3f}  {r['id']}: {r['text'][:100]}...")

print("\nFinal grounded answer:")
print(answer_with_citations(query, reranked))


## Stage 4 — Quality Gate: Great Expectations actually gates the pipeline
Runs entirely offline (no external service). Shows both the pass path and the halt-on-failure path.

In [ ]:
from src.quality.expectations_gate import run_quality_gate, QualityGateFailure
import pandas as pd

clean_df = pd.DataFrame(valid)
result = run_quality_gate(clean_df)
print(f"PASS: {result['statistics']['successful_expectations']}/{result['statistics']['evaluated_expectations']} expectations green")

dirty_df = pd.concat([clean_df, clean_df.iloc[[0]]], ignore_index=True)
dirty_df.loc[dirty_df.index[-1], "quantity"] = 999_999
try:
    run_quality_gate(dirty_df)
except QualityGateFailure as exc:
    print(f"HALT (expected): {exc}")


## Stage 5 — Lineage: real OpenLineage START/COMPLETE/FAIL events

In [ ]:
!python -m src.lineage.lineage_emitter


## Stage 6 — Orchestration: the Airflow DAG, executed by the real scheduler
`airflow dags test` runs every task through Airflow's actual TaskInstance execution path — this is not a hand-rolled orchestrator standing in for Airflow.

In [ ]:
!pip install -q "apache-airflow==2.9.3" --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.3/constraints-3.12.txt"
import os
os.environ["AIRFLOW_HOME"] = "/content/airflow_home"
os.environ["AIRFLOW__CORE__DAGS_FOLDER"] = os.path.abspath("src/orchestration")
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
!airflow db init
!airflow dags test capstone_pipeline 2026-01-01


## Summary

This run proved, end to end, with real libraries and real captured output:
- Kafka ingestion with a Pydantic contract gate and a quarantine dead-letter path
- A Delta Lake Bronze/Silver/Gold lakehouse with a genuine `MERGE` upsert and a proven
  schema-enforcement rejection
- A hybrid dense+BM25 RAG pipeline fused with Reciprocal Rank Fusion and cross-encoder
  reranked, answering with citations
- A Great Expectations quality gate that both passes clean data and **halts** on dirty data
- OpenLineage START/COMPLETE/FAIL events for every stage
- An Airflow DAG executed by the real scheduler, with `quality_gate` upstream of every
  downstream task so a failed gate stops the run before Gold or RAG indexing happens

See `docs/ARCHITECTURE.md` for the full design writeup and `README.md` for setup instructions.
